# 10 — LODO Utterance-Level Fusion Models (Revision)

This notebook trains and evaluates utterance-level models under the Leave-One-Dataset-Out (LODO) protocol.

Models included:
1. Handcrafted SVM-RBF
2. Handcrafted MLP
3. emotion2vec MLP
4. Concat Fusion MLP

The notebook writes all revision outputs to `newcode/results_lodo_utterance_fusion_plus_base`.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Imports and configuration

In [2]:
import os
import json
import pickle
import random
import hashlib
import shutil
import platform
import datetime
import importlib.metadata as im
import numpy as np
import pandas as pd

from pathlib import Path
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    recall_score,
    classification_report,
    confusion_matrix
)

SEEDS = [42, 123, 2024]

BASE_PROJECT = Path("/content/drive/MyDrive/New Jurnal Cross")

HC_DIR = BASE_PROJECT / "processed_intra_features_hc_noaug"
E2V_DIR = BASE_PROJECT / "processed_intra_features_e2v_plus_base"

OUT_DIR = BASE_PROJECT / "results_lodo_utterance_fusion_plus_base"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DATASETS = ["emodb", "ravdess", "resd"]

LABELS = ["angry", "disgust", "fear", "happy", "neutral", "sad"]
ID_TO_LABEL = {i: label for i, label in enumerate(LABELS)}
LABEL_TO_ID = {label: i for i, label in ID_TO_LABEL.items()}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("DEVICE:", DEVICE)
print("BASE_PROJECT:", BASE_PROJECT)
print("HC_DIR :", HC_DIR)
print("E2V_DIR:", E2V_DIR)
print("OUT_DIR:", OUT_DIR)

DEVICE: cuda
BASE_PROJECT: /content/drive/MyDrive/New Jurnal Cross
HC_DIR : /content/drive/MyDrive/New Jurnal Cross/processed_intra_features_hc_noaug
E2V_DIR: /content/drive/MyDrive/New Jurnal Cross/processed_intra_features_e2v_plus_base
OUT_DIR: /content/drive/MyDrive/New Jurnal Cross/results_lodo_utterance_fusion_plus_base


## 2. Utility functions

In [3]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def compute_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "uar": recall_score(y_true, y_pred, average="macro", zero_division=0),
    }


def make_report_df(y_true, y_pred):
    report = classification_report(
        y_true,
        y_pred,
        target_names=LABELS,
        labels=list(range(len(LABELS))),
        zero_division=0,
        output_dict=True,
    )
    return pd.DataFrame(report).transpose()


def save_confusion_matrix_csv(cm, out_path):
    df_cm = pd.DataFrame(cm, index=LABELS, columns=LABELS)
    df_cm.to_csv(out_path, index=True)


def make_loader(X, y, batch_size=32, shuffle=False):
    X_tensor = torch.tensor(X, dtype=torch.float32)
    y_tensor = torch.tensor(y, dtype=torch.long)
    dataset = TensorDataset(X_tensor, y_tensor)

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=False,
    )


def compute_class_weights(y_train, n_classes=6):
    counts = np.bincount(y_train, minlength=n_classes).astype(np.float32)
    weights = counts.sum() / (n_classes * counts)
    weights = weights / weights.mean()
    return torch.tensor(weights, dtype=torch.float32)


def mean_std_str(mean, std, scale=100):
    return f"{mean * scale:.2f} ± {std * scale:.2f}"


def pkg_ver(name):
    try:
        return im.version(name)
    except Exception:
        return "not installed"


def sha256_of_file(path, chunk_size=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

## 3. Reproducibility manifests for training and cached features

This section records the software environment of this LODO training run and SHA256 checksums of cached feature files.
The emotion2vec checkpoint manifest should be generated in `05_extract_emotion2vec_intra.ipynb` and copied here if available.

In [4]:
training_env_manifest = {
    "script": "10_train_lodo_utterance_fusion.ipynb",
    "base_project": str(BASE_PROJECT),
    "hc_dir": str(HC_DIR),
    "e2v_dir": str(E2V_DIR),
    "out_dir": str(OUT_DIR),
    "created_utc": datetime.datetime.utcnow().isoformat() + "Z",
    "python_version": platform.python_version(),
    "torch_version": pkg_ver("torch"),
    "numpy_version": pkg_ver("numpy"),
    "pandas_version": pkg_ver("pandas"),
    "scikit_learn_version": pkg_ver("scikit-learn"),
}

with open(OUT_DIR / "training_environment_manifest.json", "w") as f:
    json.dump(training_env_manifest, f, indent=2)

print(json.dumps(training_env_manifest, indent=2))
print("Saved:", OUT_DIR / "training_environment_manifest.json")

# Copy emotion2vec extraction environment manifest from notebook 05 if it exists.
e2v_env_manifest = E2V_DIR / "environment_manifest.json"
if e2v_env_manifest.exists():
    shutil.copy2(e2v_env_manifest, OUT_DIR / "emotion2vec_extraction_environment_manifest.json")
    print("Copied:", e2v_env_manifest)
else:
    print("Warning: emotion2vec environment_manifest.json not found:", e2v_env_manifest)

# Hash cached feature files used as input to this LODO training notebook.
feature_manifest_rows = []
input_suffixes = {".npy", ".csv", ".pkl", ".json"}

for root_dir, feature_type in [(HC_DIR, "handcrafted"), (E2V_DIR, "emotion2vec")]:
    if not root_dir.exists():
        print("Warning: input directory does not exist:", root_dir)
        continue

    for path in sorted(root_dir.rglob("*")):
        if path.is_file() and path.suffix in input_suffixes:
            feature_manifest_rows.append({
                "feature_type": feature_type,
                "relative_path": str(path.relative_to(BASE_PROJECT)),
                "absolute_path": str(path),
                "size_bytes": int(path.stat().st_size),
                "sha256": sha256_of_file(path),
            })

feature_manifest = pd.DataFrame(feature_manifest_rows)
feature_manifest.to_csv(OUT_DIR / "input_feature_file_manifest_sha256.csv", index=False)

display(feature_manifest.head())
print("Saved:", OUT_DIR / "input_feature_file_manifest_sha256.csv")
print("Number of feature/cache files hashed:", len(feature_manifest))

{
  "script": "10_train_lodo_utterance_fusion.ipynb",
  "base_project": "/content/drive/MyDrive/New Jurnal Cross",
  "hc_dir": "/content/drive/MyDrive/New Jurnal Cross/processed_intra_features_hc_noaug",
  "e2v_dir": "/content/drive/MyDrive/New Jurnal Cross/processed_intra_features_e2v_plus_base",
  "out_dir": "/content/drive/MyDrive/New Jurnal Cross/results_lodo_utterance_fusion_plus_base",
  "created_utc": "2026-08-08T21:32:50.479384Z",
  "python_version": "3.12.13",
  "torch_version": "2.11.0+cu128",
  "numpy_version": "2.0.2",
  "pandas_version": "2.2.2",
  "scikit_learn_version": "1.6.1"
}
Saved: /content/drive/MyDrive/New Jurnal Cross/results_lodo_utterance_fusion_plus_base/training_environment_manifest.json
Copied: /content/drive/MyDrive/New Jurnal Cross/processed_intra_features_e2v_plus_base/environment_manifest.json


/tmp/ipykernel_3080/391990731.py:7: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "created_utc": datetime.datetime.utcnow().isoformat() + "Z",


,feature_type,relative_path,absolute_path,size_bytes,sha256
0,handcrafted,processed_intra_features_hc_noaug/emodb/X_hc_t...,/content/drive/MyDrive/New Jurnal Cross/proces...,326736,9123372301ff31603359240e3fcb680a0e81b776c3c55e...
1,handcrafted,processed_intra_features_hc_noaug/emodb/X_hc_t...,/content/drive/MyDrive/New Jurnal Cross/proces...,1091744,61750a43b9241cdd3f8ed3fd761bb299458e6c5fb34bad...
2,handcrafted,processed_intra_features_hc_noaug/emodb/X_hc_v...,/content/drive/MyDrive/New Jurnal Cross/proces...,155760,e6e8806d6cc4e02911c77c8d1e54d65fa96919b4b22261...
3,handcrafted,processed_intra_features_hc_noaug/emodb/errors...,/content/drive/MyDrive/New Jurnal Cross/proces...,6,be64adb52ee58dc731a15683944c1b8057e8276eca0793...
4,handcrafted,processed_intra_features_hc_noaug/emodb/featur...,/content/drive/MyDrive/New Jurnal Cross/proces...,492,7804b80ec997ba23bad7ab5f7d307d243956e086c78ded...


Saved: /content/drive/MyDrive/New Jurnal Cross/results_lodo_utterance_fusion_plus_base/input_feature_file_manifest_sha256.csv
Number of feature/cache files hashed: 85


## 4. Load raw per-dataset features

Important: the handcrafted feature files are intra-corpus scaled. This notebook recovers raw handcrafted features with the original intra-corpus scaler and then fits a new LODO scaler only on source training data.

In [5]:
def load_one_dataset_raw_features(dataset_name):
    """
    Load per-dataset features.

    emotion2vec:
      raw embedding, not scaled.

    handcrafted:
      X_hc_*.npy was previously intra-corpus scaled.
      We recover the raw values using scaler_hc.pkl.
    """
    hc_ds = HC_DIR / dataset_name
    e2v_ds = E2V_DIR / dataset_name

    required_paths = [
        hc_ds / "X_hc_train.npy",
        hc_ds / "X_hc_val.npy",
        hc_ds / "X_hc_test.npy",
        hc_ds / "scaler_hc.pkl",
        e2v_ds / "X_e2v_train.npy",
        e2v_ds / "X_e2v_val.npy",
        e2v_ds / "X_e2v_test.npy",
        e2v_ds / "y_train.npy",
        e2v_ds / "y_val.npy",
        e2v_ds / "y_test.npy",
        e2v_ds / "meta_train.csv",
        e2v_ds / "meta_val.csv",
        e2v_ds / "meta_test.csv",
    ]
    missing = [str(p) for p in required_paths if not p.exists()]
    if missing:
        raise FileNotFoundError("Missing required files:\n" + "\n".join(missing))
    # Load intra-scaled handcrafted features
    X_hc_train_scaled = np.load(hc_ds / "X_hc_train.npy").astype(np.float32)
    X_hc_val_scaled = np.load(hc_ds / "X_hc_val.npy").astype(np.float32)
    X_hc_test_scaled = np.load(hc_ds / "X_hc_test.npy").astype(np.float32)

    with open(hc_ds / "scaler_hc.pkl", "rb") as f:
        intra_scaler_hc = pickle.load(f)

    # Recover raw handcrafted features
    X_hc_train = intra_scaler_hc.inverse_transform(X_hc_train_scaled).astype(np.float32)
    X_hc_val = intra_scaler_hc.inverse_transform(X_hc_val_scaled).astype(np.float32)
    X_hc_test = intra_scaler_hc.inverse_transform(X_hc_test_scaled).astype(np.float32)

    # Load raw emotion2vec embedding
    X_e2v_train = np.load(e2v_ds / "X_e2v_train.npy").astype(np.float32)
    X_e2v_val = np.load(e2v_ds / "X_e2v_val.npy").astype(np.float32)
    X_e2v_test = np.load(e2v_ds / "X_e2v_test.npy").astype(np.float32)

    y_train = np.load(e2v_ds / "y_train.npy").astype(np.int64)
    y_val = np.load(e2v_ds / "y_val.npy").astype(np.int64)
    y_test = np.load(e2v_ds / "y_test.npy").astype(np.int64)

    meta_train = pd.read_csv(e2v_ds / "meta_train.csv")
    meta_val = pd.read_csv(e2v_ds / "meta_val.csv")
    meta_test = pd.read_csv(e2v_ds / "meta_test.csv")

    # Safety check
    assert X_hc_train.shape[0] == X_e2v_train.shape[0] == len(y_train) == len(meta_train)
    assert X_hc_val.shape[0] == X_e2v_val.shape[0] == len(y_val) == len(meta_val)
    assert X_hc_test.shape[0] == X_e2v_test.shape[0] == len(y_test) == len(meta_test)

    return {
        "X_hc_train": X_hc_train,
        "X_hc_val": X_hc_val,
        "X_hc_test": X_hc_test,

        "X_e2v_train": X_e2v_train,
        "X_e2v_val": X_e2v_val,
        "X_e2v_test": X_e2v_test,

        "y_train": y_train,
        "y_val": y_val,
        "y_test": y_test,

        "meta_train": meta_train,
        "meta_val": meta_val,
        "meta_test": meta_test,
    }


dataset_cache = {}

for ds in DATASETS:
    dataset_cache[ds] = load_one_dataset_raw_features(ds)
    print("=" * 80)
    print(ds.upper())
    print("HC train :", dataset_cache[ds]["X_hc_train"].shape)
    print("E2V train:", dataset_cache[ds]["X_e2v_train"].shape)
    print("y train  :", dataset_cache[ds]["y_train"].shape)

EMODB
HC train : (498, 548)
E2V train: (498, 768)
y train  : (498,)
RAVDESS
HC train : (704, 548)
E2V train: (704, 768)
y train  : (704,)
RESD
HC train : (873, 548)
E2V train: (873, 768)
y train  : (873,)


## 4. Construct LODO folds

Protocol:
- Source train + source test are merged into source-domain training data.
- Source validation is used for validation and model selection.
- The full held-out target corpus is used only for final testing.
- StandardScaler is fitted only on LODO source training data.

In [6]:
def concat_parts(parts):
    return np.concatenate(parts, axis=0)


def add_role_columns(meta, corpus, original_split, role, fold_name):
    m = meta.copy()
    m.insert(0, "fold", fold_name)
    m.insert(1, "corpus", corpus)
    m.insert(2, "original_split", original_split)
    m.insert(3, "role", role)
    return m


def make_lodo_fold(test_dataset):
    """
    Train:
      train + test splits from source datasets

    Validation:
      validation splits from source datasets

    Test:
      train + validation + test splits from target dataset
    """
    source_datasets = [ds for ds in DATASETS if ds != test_dataset]
    fold_name = f"heldout_{test_dataset}"

    # Train from source train + source test
    X_hc_train = concat_parts(
        [dataset_cache[ds]["X_hc_train"] for ds in source_datasets] +
        [dataset_cache[ds]["X_hc_test"] for ds in source_datasets]
    )

    X_e2v_train = concat_parts(
        [dataset_cache[ds]["X_e2v_train"] for ds in source_datasets] +
        [dataset_cache[ds]["X_e2v_test"] for ds in source_datasets]
    )

    y_train = concat_parts(
        [dataset_cache[ds]["y_train"] for ds in source_datasets] +
        [dataset_cache[ds]["y_test"] for ds in source_datasets]
    )

    meta_train = pd.concat(
        [add_role_columns(dataset_cache[ds]["meta_train"], ds, "train", "source_train", fold_name) for ds in source_datasets] +
        [add_role_columns(dataset_cache[ds]["meta_test"], ds, "test", "source_train", fold_name) for ds in source_datasets],
        ignore_index=True,
    )

    # Validation from source val
    X_hc_val = concat_parts([dataset_cache[ds]["X_hc_val"] for ds in source_datasets])
    X_e2v_val = concat_parts([dataset_cache[ds]["X_e2v_val"] for ds in source_datasets])
    y_val = concat_parts([dataset_cache[ds]["y_val"] for ds in source_datasets])
    meta_val = pd.concat(
        [add_role_columns(dataset_cache[ds]["meta_val"], ds, "val", "source_validation", fold_name) for ds in source_datasets],
        ignore_index=True,
    )

    # Target test = all target splits
    target = dataset_cache[test_dataset]

    X_hc_test = concat_parts([
        target["X_hc_train"],
        target["X_hc_val"],
        target["X_hc_test"],
    ])

    X_e2v_test = concat_parts([
        target["X_e2v_train"],
        target["X_e2v_val"],
        target["X_e2v_test"],
    ])

    y_test = concat_parts([
        target["y_train"],
        target["y_val"],
        target["y_test"],
    ])

    meta_test = pd.concat(
        [
            add_role_columns(target["meta_train"], test_dataset, "train", "target_test", fold_name),
            add_role_columns(target["meta_val"], test_dataset, "val", "target_test", fold_name),
            add_role_columns(target["meta_test"], test_dataset, "test", "target_test", fold_name),
        ],
        ignore_index=True,
    )

    assert len(meta_train) == len(y_train) == X_hc_train.shape[0] == X_e2v_train.shape[0]
    assert len(meta_val) == len(y_val) == X_hc_val.shape[0] == X_e2v_val.shape[0]
    assert len(meta_test) == len(y_test) == X_hc_test.shape[0] == X_e2v_test.shape[0]

    # Fit scalers only on LODO source training data
    scaler_hc = StandardScaler()
    X_hc_train_scaled = scaler_hc.fit_transform(X_hc_train).astype(np.float32)
    X_hc_val_scaled = scaler_hc.transform(X_hc_val).astype(np.float32)
    X_hc_test_scaled = scaler_hc.transform(X_hc_test).astype(np.float32)

    scaler_e2v = StandardScaler()
    X_e2v_train_scaled = scaler_e2v.fit_transform(X_e2v_train).astype(np.float32)
    X_e2v_val_scaled = scaler_e2v.transform(X_e2v_val).astype(np.float32)
    X_e2v_test_scaled = scaler_e2v.transform(X_e2v_test).astype(np.float32)

    X_concat_train = np.concatenate([X_e2v_train_scaled, X_hc_train_scaled], axis=1).astype(np.float32)
    X_concat_val = np.concatenate([X_e2v_val_scaled, X_hc_val_scaled], axis=1).astype(np.float32)
    X_concat_test = np.concatenate([X_e2v_test_scaled, X_hc_test_scaled], axis=1).astype(np.float32)

    return {
        "fold_name": fold_name,
        "test_dataset": test_dataset,
        "source_datasets": source_datasets,

        "X_e2v_train": X_e2v_train_scaled,
        "X_e2v_val": X_e2v_val_scaled,
        "X_e2v_test": X_e2v_test_scaled,

        "X_hc_train": X_hc_train_scaled,
        "X_hc_val": X_hc_val_scaled,
        "X_hc_test": X_hc_test_scaled,

        "X_concat_train": X_concat_train,
        "X_concat_val": X_concat_val,
        "X_concat_test": X_concat_test,

        "y_train": y_train,
        "y_val": y_val,
        "y_test": y_test,

        "meta_train": meta_train,
        "meta_val": meta_val,
        "meta_test": meta_test,

        "scaler_hc": scaler_hc,
        "scaler_e2v": scaler_e2v,
    }

## 5. Save split manifest and leakage-prevention checklist

In [7]:
manifest_parts = []
summary_parts = []

for test_ds in DATASETS:
    fold = make_lodo_fold(test_ds)

    manifest_parts.extend([fold["meta_train"], fold["meta_val"], fold["meta_test"]])

    for role_name, meta_df, y_arr in [
        ("source_train", fold["meta_train"], fold["y_train"]),
        ("source_validation", fold["meta_val"], fold["y_val"]),
        ("target_test", fold["meta_test"], fold["y_test"]),
    ]:
        counts = pd.Series(y_arr).value_counts().sort_index().rename(index=ID_TO_LABEL)
        row = {
            "fold": fold["fold_name"],
            "test_dataset": test_ds,
            "source_datasets": "+".join(fold["source_datasets"]),
            "role": role_name,
            "n": len(meta_df),
        }
        for label in LABELS:
            row[f"n_{label}"] = int(counts.get(label, 0))
        summary_parts.append(row)

manifest = pd.concat(manifest_parts, ignore_index=True)
manifest.to_csv(OUT_DIR / "lodo_split_manifest.csv", index=False)

split_summary = pd.DataFrame(summary_parts)
split_summary.to_csv(OUT_DIR / "lodo_split_summary.csv", index=False)

leakage_checklist = pd.DataFrame([
    {
        "component": "LODO fold construction",
        "procedure": "Target corpus train, validation, and test splits are merged only as target_test.",
        "target_used_for_training_or_selection": "No",
    },
    {
        "component": "Handcrafted feature scaling",
        "procedure": "StandardScaler is fitted only on LODO source_train handcrafted features and then applied to source_validation and target_test.",
        "target_used_for_training_or_selection": "No",
    },
    {
        "component": "emotion2vec feature scaling",
        "procedure": "StandardScaler is fitted only on LODO source_train emotion2vec embeddings and then applied to source_validation and target_test.",
        "target_used_for_training_or_selection": "No",
    },
    {
        "component": "Class weighting",
        "procedure": "Class weights are computed only from source_train labels.",
        "target_used_for_training_or_selection": "No",
    },
    {
        "component": "Early stopping",
        "procedure": "Best epoch is selected using source_validation Macro-F1 only.",
        "target_used_for_training_or_selection": "No",
    },
    {
        "component": "Final target evaluation",
        "procedure": "Target labels are used only for final metric computation after model selection.",
        "target_used_for_training_or_selection": "Evaluation only",
    },
])
leakage_checklist.to_csv(OUT_DIR / "lodo_leakage_prevention_checklist.csv", index=False)

print("Saved:", OUT_DIR / "lodo_split_manifest.csv")
print("Saved:", OUT_DIR / "lodo_split_summary.csv")
print("Saved:", OUT_DIR / "lodo_leakage_prevention_checklist.csv")

# Optional overlap/duplicate check based on any file-level identifier columns present in metadata.
# This cannot replace audio-level hashing, but it documents whether the same known file identifiers occur in multiple LODO roles.
candidate_id_cols = [
    "file_id", "utt_id", "utterance_id", "audio_id",
    "filepath", "file_path", "path", "audio_path", "wav_path",
    "filename", "file", "name",
]
available_id_cols = [c for c in candidate_id_cols if c in manifest.columns]

overlap_rows = []
for fold_name, fold_df in manifest.groupby("fold"):
    for id_col in available_id_cols:
        role_sets = {
            role: set(
                fold_df.loc[fold_df["role"] == role, id_col]
                .dropna()
                .astype(str)
                .tolist()
            )
            for role in ["source_train", "source_validation", "target_test"]
        }

        comparisons = [
            ("source_train", "source_validation"),
            ("source_train", "target_test"),
            ("source_validation", "target_test"),
        ]

        for a, b in comparisons:
            inter = role_sets[a].intersection(role_sets[b])
            overlap_rows.append({
                "fold": fold_name,
                "identifier_column": id_col,
                "role_a": a,
                "role_b": b,
                "n_overlap": len(inter),
                "example_overlap_values": "; ".join(list(sorted(inter))[:5]),
            })

if overlap_rows:
    overlap_check = pd.DataFrame(overlap_rows)
else:
    overlap_check = pd.DataFrame([{
        "fold": "all",
        "identifier_column": "none_found",
        "role_a": "not_applicable",
        "role_b": "not_applicable",
        "n_overlap": np.nan,
        "example_overlap_values": "No file-level identifier columns were found in metadata.",
    }])

overlap_check.to_csv(OUT_DIR / "lodo_partition_overlap_check.csv", index=False)
print("Saved:", OUT_DIR / "lodo_partition_overlap_check.csv")

display(split_summary)
display(leakage_checklist)
display(overlap_check)

for test_ds in DATASETS:
    fold = make_lodo_fold(test_ds)
    print("=" * 80)
    print("TEST:", test_ds)
    print("SOURCE:", fold["source_datasets"])
    print("E2V train:", fold["X_e2v_train"].shape)
    print("HC train:", fold["X_hc_train"].shape)
    print("Concat train:", fold["X_concat_train"].shape)
    print("Val:", fold["X_concat_val"].shape)
    print("Test:", fold["X_concat_test"].shape)
    print("Test label distribution:")
    print(pd.Series(fold["y_test"]).value_counts().sort_index().rename(index=ID_TO_LABEL))

Saved: /content/drive/MyDrive/New Jurnal Cross/results_lodo_utterance_fusion_plus_base/lodo_split_manifest.csv
Saved: /content/drive/MyDrive/New Jurnal Cross/results_lodo_utterance_fusion_plus_base/lodo_split_summary.csv
Saved: /content/drive/MyDrive/New Jurnal Cross/results_lodo_utterance_fusion_plus_base/lodo_leakage_prevention_checklist.csv
Saved: /content/drive/MyDrive/New Jurnal Cross/results_lodo_utterance_fusion_plus_base/lodo_partition_overlap_check.csv


,fold,test_dataset,source_datasets,role,n,n_angry,n_disgust,n_fear,n_happy,n_neutral,n_sad
0,heldout_emodb,emodb,ravdess+resd,source_train,1892,348,310,351,343,240,300
1,heldout_emodb,emodb,ravdess+resd,source_validation,362,63,67,64,67,47,54
2,heldout_emodb,emodb,ravdess+resd,target_test,718,140,106,123,118,106,125
3,heldout_ravdess,ravdess,emodb+resd,source_train,1659,315,245,301,289,256,253
4,heldout_ravdess,ravdess,emodb+resd,source_validation,257,44,46,45,47,41,34
5,heldout_ravdess,ravdess,emodb+resd,target_test,1056,192,192,192,192,96,192
6,heldout_resd,resd,emodb+ravdess,source_train,1527,287,255,270,266,176,273
7,heldout_resd,resd,emodb+ravdess,source_validation,247,45,43,45,44,26,44
8,heldout_resd,resd,emodb+ravdess,target_test,1198,219,185,223,218,191,162


,component,procedure,target_used_for_training_or_selection
0,LODO fold construction,"Target corpus train, validation, and test spli...",No
1,Handcrafted feature scaling,StandardScaler is fitted only on LODO source_t...,No
2,emotion2vec feature scaling,StandardScaler is fitted only on LODO source_t...,No
3,Class weighting,Class weights are computed only from source_tr...,No
4,Early stopping,Best epoch is selected using source_validation...,No
5,Final target evaluation,Target labels are used only for final metric c...,Evaluation only


,fold,identifier_column,role_a,role_b,n_overlap,example_overlap_values
0,heldout_emodb,filepath,source_train,source_validation,0,
1,heldout_emodb,filepath,source_train,target_test,0,
2,heldout_emodb,filepath,source_validation,target_test,0,
3,heldout_emodb,filename,source_train,source_validation,0,
4,heldout_emodb,filename,source_train,target_test,0,
5,heldout_emodb,filename,source_validation,target_test,0,
6,heldout_ravdess,filepath,source_train,source_validation,0,
7,heldout_ravdess,filepath,source_train,target_test,0,
8,heldout_ravdess,filepath,source_validation,target_test,0,
9,heldout_ravdess,filename,source_train,source_validation,0,


TEST: emodb
SOURCE: ['ravdess', 'resd']
E2V train: (1892, 768)
HC train: (1892, 548)
Concat train: (1892, 1316)
Val: (362, 1316)
Test: (718, 1316)
Test label distribution:
angry      140
disgust    106
fear       123
happy      118
neutral    106
sad        125
Name: count, dtype: int64
TEST: ravdess
SOURCE: ['emodb', 'resd']
E2V train: (1659, 768)
HC train: (1659, 548)
Concat train: (1659, 1316)
Val: (257, 1316)
Test: (1056, 1316)
Test label distribution:
angry      192
disgust    192
fear       192
happy      192
neutral     96
sad        192
Name: count, dtype: int64
TEST: resd
SOURCE: ['emodb', 'ravdess']
E2V train: (1527, 768)
HC train: (1527, 548)
Concat train: (1527, 1316)
Val: (247, 1316)
Test: (1198, 1316)
Test label distribution:
angry      219
disgust    185
fear       223
happy      218
neutral    191
sad        162
Name: count, dtype: int64


## 6. Neural model definitions

In [8]:
class SimpleMLP(nn.Module):
    """LayerNorm MLP used for emotion2vec utterance-level embeddings."""
    def __init__(
        self,
        input_dim,
        hidden_dim=256,
        num_classes=6,
        dropout=0.30,
    ):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim // 2, num_classes),
        )

    def forward(self, x):
        return self.net(x)


class HandcraftedMLP(nn.Module):
    """BatchNorm MLP used for the handcrafted 548-dimensional vector."""
    def __init__(
        self,
        input_dim,
        hidden_dim=256,
        num_classes=6,
        dropout=0.30,
    ):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim // 2, num_classes),
        )

    def forward(self, x):
        return self.net(x)


class ConcatFusionMLP(nn.Module):
    def __init__(
        self,
        input_dim,
        hidden_dim=512,
        num_classes=6,
        dropout=0.35,
    ):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim // 2, hidden_dim // 4),
            nn.LayerNorm(hidden_dim // 4),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim // 4, num_classes),
        )

    def forward(self, x):
        return self.net(x)

## 7. Neural training and prediction functions

In [9]:
def run_one_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None

    if is_train:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    all_preds = []
    all_targets = []

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)

        if is_train:
            optimizer.zero_grad()

        with torch.set_grad_enabled(is_train):
            logits = model(X_batch)
            loss = criterion(logits, y_batch)

            if is_train:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
                optimizer.step()

        total_loss += loss.item() * X_batch.size(0)

        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.detach().cpu().numpy().tolist())
        all_targets.extend(y_batch.detach().cpu().numpy().tolist())

    avg_loss = total_loss / len(loader.dataset)
    metrics = compute_metrics(np.array(all_targets), np.array(all_preds))

    return avg_loss, metrics


@torch.no_grad()
def predict_model(model, loader):
    model.eval()

    all_preds = []
    all_targets = []
    all_probs = []

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(DEVICE)

        logits = model(X_batch)
        probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(probs, dim=1)

        all_preds.extend(preds.cpu().numpy().tolist())
        all_targets.extend(y_batch.numpy().tolist())
        all_probs.extend(probs.cpu().numpy().tolist())

    return np.array(all_targets), np.array(all_preds), np.array(all_probs)

## 8. Train and evaluate MLP-based LODO models

In [10]:
def train_eval_lodo_mlp(
    fold,
    model_name,
    seed,
    batch_size=32,
    lr=1e-3,
    weight_decay=1e-4,
    max_epochs=150,
    patience=20,
):
    set_seed(seed)

    if model_name == "emotion2vec MLP":
        X_train = fold["X_e2v_train"]
        X_val = fold["X_e2v_val"]
        X_test = fold["X_e2v_test"]

        model = SimpleMLP(
            input_dim=X_train.shape[1],
            hidden_dim=256,
            num_classes=len(LABELS),
            dropout=0.30,
        ).to(DEVICE)

    elif model_name == "Concat Fusion MLP":
        X_train = fold["X_concat_train"]
        X_val = fold["X_concat_val"]
        X_test = fold["X_concat_test"]

        model = ConcatFusionMLP(
            input_dim=X_train.shape[1],
            hidden_dim=512,
            num_classes=len(LABELS),
            dropout=0.35,
        ).to(DEVICE)

    elif model_name == "Handcrafted MLP":
        X_train = fold["X_hc_train"]
        X_val = fold["X_hc_val"]
        X_test = fold["X_hc_test"]

        model = HandcraftedMLP(
            input_dim=X_train.shape[1],
            hidden_dim=256,
            num_classes=len(LABELS),
            dropout=0.30,
        ).to(DEVICE)

    else:
        raise ValueError(f"Unknown model_name: {model_name}")

    y_train = fold["y_train"]
    y_val = fold["y_val"]
    y_test = fold["y_test"]

    train_loader = make_loader(X_train, y_train, batch_size=batch_size, shuffle=True)
    val_loader = make_loader(X_val, y_val, batch_size=batch_size, shuffle=False)
    test_loader = make_loader(X_test, y_test, batch_size=batch_size, shuffle=False)

    class_weights = compute_class_weights(y_train, n_classes=len(LABELS)).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay,
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=5,
    )

    best_val_macro_f1 = -1.0
    best_epoch = -1
    best_state = None
    no_improve = 0
    history = []

    for epoch in range(1, max_epochs + 1):
        train_loss, train_metrics = run_one_epoch(
            model,
            train_loader,
            criterion,
            optimizer=optimizer,
        )

        val_loss, val_metrics = run_one_epoch(
            model,
            val_loader,
            criterion,
            optimizer=None,
        )

        scheduler.step(val_metrics["macro_f1"])

        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            **{f"train_{k}": v for k, v in train_metrics.items()},
            **{f"val_{k}": v for k, v in val_metrics.items()},
            "lr": optimizer.param_groups[0]["lr"],
        }
        history.append(row)

        current = val_metrics["macro_f1"]

        if current > best_val_macro_f1:
            best_val_macro_f1 = current
            best_epoch = epoch
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
            no_improve = 0
        else:
            no_improve += 1

        if epoch % 10 == 0 or epoch == 1:
            print(
                f"[Test={fold['test_dataset']} | {model_name} | seed={seed}] "
                f"Epoch {epoch:03d} | "
                f"val_macro_f1={val_metrics['macro_f1']:.4f} | "
                f"best={best_val_macro_f1:.4f}"
            )

        if no_improve >= patience:
            print(
                f"[Test={fold['test_dataset']} | {model_name} | seed={seed}] "
                f"Early stopping at epoch {epoch}. Best epoch={best_epoch}"
            )
            break

    if best_state is None:
        raise RuntimeError("No best model state was stored. Check validation loop.")

    model.load_state_dict(best_state)

    y_val_true, y_val_pred, y_val_prob = predict_model(model, val_loader)
    y_test_true, y_test_pred, y_test_prob = predict_model(model, test_loader)

    val_metrics = compute_metrics(y_val_true, y_val_pred)
    test_metrics = compute_metrics(y_test_true, y_test_pred)

    safe_model_name = model_name.lower().replace(" ", "_").replace("-", "_")
    run_dir = OUT_DIR / safe_model_name / f"test_{fold['test_dataset']}" / f"seed_{seed}"
    run_dir.mkdir(parents=True, exist_ok=True)

    torch.save(model.state_dict(), run_dir / "model.pt")
    pd.DataFrame(history).to_csv(run_dir / "training_history.csv", index=False)

    with open(run_dir / "training_config.json", "w") as f:
        json.dump({
            "model_name": model_name,
            "test_dataset": fold["test_dataset"],
            "source_datasets": fold["source_datasets"],
            "seed": seed,
            "best_epoch": int(best_epoch),
            "best_val_macro_f1": float(best_val_macro_f1),
            "batch_size": batch_size,
            "lr": lr,
            "weight_decay": weight_decay,
            "max_epochs": max_epochs,
            "patience": patience,
            "input_dim": int(X_train.shape[1]),
            "num_classes": len(LABELS),
        }, f, indent=2)

    # Save scalers for reproducibility
    with open(run_dir / "scaler_hc_lodo.pkl", "wb") as f:
        pickle.dump(fold["scaler_hc"], f)

    with open(run_dir / "scaler_e2v_lodo.pkl", "wb") as f:
        pickle.dump(fold["scaler_e2v"], f)

    # Reports
    make_report_df(y_val_true, y_val_pred).to_csv(run_dir / "val_classification_report.csv")
    make_report_df(y_test_true, y_test_pred).to_csv(run_dir / "test_classification_report.csv")

    cm_val = confusion_matrix(y_val_true, y_val_pred, labels=list(range(len(LABELS))))
    cm_test = confusion_matrix(y_test_true, y_test_pred, labels=list(range(len(LABELS))))

    save_confusion_matrix_csv(cm_val, run_dir / "val_confusion_matrix.csv")
    save_confusion_matrix_csv(cm_test, run_dir / "test_confusion_matrix.csv")

    pred_val_df = fold["meta_val"].copy()
    pred_val_df["y_true"] = y_val_true
    pred_val_df["y_pred"] = y_val_pred
    pred_val_df["true_label"] = [ID_TO_LABEL[i] for i in y_val_true]
    pred_val_df["pred_label"] = [ID_TO_LABEL[i] for i in y_val_pred]
    for i, label in enumerate(LABELS):
        pred_val_df[f"prob_{label}"] = y_val_prob[:, i]
    pred_val_df.to_csv(run_dir / "val_predictions.csv", index=False)

    pred_test_df = fold["meta_test"].copy()
    pred_test_df["y_true"] = y_test_true
    pred_test_df["y_pred"] = y_test_pred
    pred_test_df["true_label"] = [ID_TO_LABEL[i] for i in y_test_true]
    pred_test_df["pred_label"] = [ID_TO_LABEL[i] for i in y_test_pred]
    for i, label in enumerate(LABELS):
        pred_test_df[f"prob_{label}"] = y_test_prob[:, i]
    pred_test_df.to_csv(run_dir / "test_predictions.csv", index=False)

    row_val = {
        "model": model_name,
        "test_dataset": fold["test_dataset"],
        "seed": seed,
        "split": "val",
        "best_epoch": best_epoch,
        "best_val_macro_f1": best_val_macro_f1,
        **val_metrics,
    }

    row_test = {
        "model": model_name,
        "test_dataset": fold["test_dataset"],
        "seed": seed,
        "split": "test",
        "best_epoch": best_epoch,
        "best_val_macro_f1": best_val_macro_f1,
        **test_metrics,
    }

    return row_val, row_test

## 9. Run MLP-based LODO models

In [11]:
mlp_rows = []

MLP_MODELS = [
    "emotion2vec MLP",
    "Concat Fusion MLP",
    "Handcrafted MLP",
]

for test_dataset in DATASETS:
    fold = make_lodo_fold(test_dataset)

    print("=" * 100)
    print(f"LODO TEST DATASET: {test_dataset.upper()}")
    print(f"SOURCE DATASETS  : {fold['source_datasets']}")
    print("=" * 100)

    for model_name in MLP_MODELS:
        for seed in SEEDS:
            print(f"\nTraining | test={test_dataset} | model={model_name} | seed={seed}")

            if model_name == "Concat Fusion MLP":
                lr = 8e-4
            else:
                lr = 1e-3

            row_val, row_test = train_eval_lodo_mlp(
                fold=fold,
                model_name=model_name,
                seed=seed,
                batch_size=32,
                lr=lr,
                weight_decay=1e-4,
                max_epochs=150,
                patience=20,
            )

            mlp_rows.append(row_val)
            mlp_rows.append(row_test)

            print("VAL :", {k: round(v, 4) for k, v in row_val.items() if isinstance(v, float)})
            print("TEST:", {k: round(v, 4) for k, v in row_test.items() if isinstance(v, float)})

mlp_results = pd.DataFrame(mlp_rows)
mlp_results.to_csv(OUT_DIR / "lodo_mlp_all_seed_results.csv", index=False)

# Backward-compatible filename for MLP-only results
mlp_results.to_csv(OUT_DIR / "lodo_all_seed_results.csv", index=False)

display(mlp_results)
print("Saved:", OUT_DIR / "lodo_mlp_all_seed_results.csv")
print("Saved:", OUT_DIR / "lodo_all_seed_results.csv")

LODO TEST DATASET: EMODB
SOURCE DATASETS  : ['ravdess', 'resd']

Training | test=emodb | model=emotion2vec MLP | seed=42
[Test=emodb | emotion2vec MLP | seed=42] Epoch 001 | val_macro_f1=0.7657 | best=0.7657
[Test=emodb | emotion2vec MLP | seed=42] Epoch 010 | val_macro_f1=0.7771 | best=0.7771
[Test=emodb | emotion2vec MLP | seed=42] Epoch 020 | val_macro_f1=0.7687 | best=0.7806
[Test=emodb | emotion2vec MLP | seed=42] Epoch 030 | val_macro_f1=0.7743 | best=0.7806
[Test=emodb | emotion2vec MLP | seed=42] Early stopping at epoch 39. Best epoch=19
VAL : {'best_val_macro_f1': 0.7806, 'accuracy': 0.7818, 'macro_f1': 0.7806, 'weighted_f1': 0.7829, 'uar': 0.786}
TEST: {'best_val_macro_f1': 0.7806, 'accuracy': 0.8357, 'macro_f1': 0.8312, 'weighted_f1': 0.832, 'uar': 0.8338}

Training | test=emodb | model=emotion2vec MLP | seed=123
[Test=emodb | emotion2vec MLP | seed=123] Epoch 001 | val_macro_f1=0.7667 | best=0.7667
[Test=emodb | emotion2vec MLP | seed=123] Epoch 010 | val_macro_f1=0.7658 | 

,model,test_dataset,seed,split,best_epoch,best_val_macro_f1,accuracy,macro_f1,weighted_f1,uar
0,emotion2vec MLP,emodb,42,val,19,0.780625,0.781768,0.780625,0.782904,0.786041
1,emotion2vec MLP,emodb,42,test,19,0.780625,0.835655,0.831243,0.831998,0.833800
2,emotion2vec MLP,emodb,123,val,6,0.786820,0.787293,0.786820,0.786547,0.789345
3,emotion2vec MLP,emodb,123,test,6,0.786820,0.845404,0.841127,0.842525,0.843851
4,emotion2vec MLP,emodb,2024,val,2,0.780464,0.781768,0.780464,0.783300,0.785326
5,emotion2vec MLP,emodb,2024,test,2,0.780464,0.838440,0.833213,0.834349,0.836536
6,Concat Fusion MLP,emodb,42,val,5,0.766035,0.767956,0.766035,0.768199,0.774703
7,Concat Fusion MLP,emodb,42,test,5,0.766035,0.821727,0.813629,0.815453,0.819464
8,Concat Fusion MLP,emodb,123,val,2,0.781237,0.781768,0.781237,0.784674,0.785537
9,Concat Fusion MLP,emodb,123,test,2,0.781237,0.832869,0.827056,0.828322,0.831791


Saved: /content/drive/MyDrive/New Jurnal Cross/results_lodo_utterance_fusion_plus_base/lodo_mlp_all_seed_results.csv
Saved: /content/drive/MyDrive/New Jurnal Cross/results_lodo_utterance_fusion_plus_base/lodo_all_seed_results.csv


## 10. Train and evaluate Handcrafted SVM-RBF under LODO

In [12]:
def train_eval_lodo_svm(fold, seed):
    set_seed(seed)

    model = SVC(
        kernel="rbf",
        C=10.0,
        gamma="scale",
        class_weight="balanced",
        probability=True,
        random_state=seed,
    )

    model.fit(fold["X_hc_train"], fold["y_train"])

    y_val_pred = model.predict(fold["X_hc_val"])
    y_test_pred = model.predict(fold["X_hc_test"])

    y_val_prob = model.predict_proba(fold["X_hc_val"])
    y_test_prob = model.predict_proba(fold["X_hc_test"])

    val_metrics = compute_metrics(fold["y_val"], y_val_pred)
    test_metrics = compute_metrics(fold["y_test"], y_test_pred)

    run_dir = OUT_DIR / "handcrafted_svm_rbf" / f"test_{fold['test_dataset']}" / f"seed_{seed}"
    run_dir.mkdir(parents=True, exist_ok=True)

    with open(run_dir / "svm_model.pkl", "wb") as f:
        pickle.dump(model, f)

    with open(run_dir / "training_config.json", "w") as f:
        json.dump({
            "model_name": "Handcrafted SVM-RBF",
            "test_dataset": fold["test_dataset"],
            "source_datasets": fold["source_datasets"],
            "seed": seed,
            "kernel": "rbf",
            "C": 10.0,
            "gamma": "scale",
            "class_weight": "balanced",
            "probability": True,
            "input_dim": int(fold["X_hc_train"].shape[1]),
            "num_classes": len(LABELS),
        }, f, indent=2)

    # Save scaler for reproducibility
    with open(run_dir / "scaler_hc_lodo.pkl", "wb") as f:
        pickle.dump(fold["scaler_hc"], f)

    # Reports
    make_report_df(fold["y_val"], y_val_pred).to_csv(run_dir / "val_classification_report.csv")
    make_report_df(fold["y_test"], y_test_pred).to_csv(run_dir / "test_classification_report.csv")

    cm_val = confusion_matrix(fold["y_val"], y_val_pred, labels=list(range(len(LABELS))))
    cm_test = confusion_matrix(fold["y_test"], y_test_pred, labels=list(range(len(LABELS))))

    save_confusion_matrix_csv(cm_val, run_dir / "val_confusion_matrix.csv")
    save_confusion_matrix_csv(cm_test, run_dir / "test_confusion_matrix.csv")

    pred_val_df = fold["meta_val"].copy()
    pred_val_df["y_true"] = fold["y_val"]
    pred_val_df["y_pred"] = y_val_pred
    pred_val_df["true_label"] = [ID_TO_LABEL[i] for i in fold["y_val"]]
    pred_val_df["pred_label"] = [ID_TO_LABEL[i] for i in y_val_pred]
    for i, label in enumerate(LABELS):
        pred_val_df[f"prob_{label}"] = y_val_prob[:, i]
    pred_val_df.to_csv(run_dir / "val_predictions.csv", index=False)

    pred_test_df = fold["meta_test"].copy()
    pred_test_df["y_true"] = fold["y_test"]
    pred_test_df["y_pred"] = y_test_pred
    pred_test_df["true_label"] = [ID_TO_LABEL[i] for i in fold["y_test"]]
    pred_test_df["pred_label"] = [ID_TO_LABEL[i] for i in y_test_pred]
    for i, label in enumerate(LABELS):
        pred_test_df[f"prob_{label}"] = y_test_prob[:, i]
    pred_test_df.to_csv(run_dir / "test_predictions.csv", index=False)

    row_val = {
        "model": "Handcrafted SVM-RBF",
        "test_dataset": fold["test_dataset"],
        "seed": seed,
        "split": "val",
        "best_epoch": np.nan,
        "best_val_macro_f1": val_metrics["macro_f1"],
        **val_metrics,
    }

    row_test = {
        "model": "Handcrafted SVM-RBF",
        "test_dataset": fold["test_dataset"],
        "seed": seed,
        "split": "test",
        "best_epoch": np.nan,
        "best_val_macro_f1": val_metrics["macro_f1"],
        **test_metrics,
    }

    return row_val, row_test


svm_rows = []

for test_dataset in DATASETS:
    fold = make_lodo_fold(test_dataset)

    print("=" * 100)
    print(f"LODO TEST DATASET: {test_dataset.upper()} | MODEL: Handcrafted SVM-RBF")
    print(f"SOURCE DATASETS  : {fold['source_datasets']}")
    print("=" * 100)

    for seed in SEEDS:
        print(f"\nTraining | test={test_dataset} | model=Handcrafted SVM-RBF | seed={seed}")

        row_val, row_test = train_eval_lodo_svm(fold, seed)

        svm_rows.append(row_val)
        svm_rows.append(row_test)

        print("VAL :", {k: round(v, 4) for k, v in row_val.items() if isinstance(v, float)})
        print("TEST:", {k: round(v, 4) for k, v in row_test.items() if isinstance(v, float)})

svm_results = pd.DataFrame(svm_rows)
svm_results.to_csv(OUT_DIR / "lodo_svm_results.csv", index=False)

display(svm_results)
print("Saved:", OUT_DIR / "lodo_svm_results.csv")

LODO TEST DATASET: EMODB | MODEL: Handcrafted SVM-RBF
SOURCE DATASETS  : ['ravdess', 'resd']

Training | test=emodb | model=Handcrafted SVM-RBF | seed=42
VAL : {'best_epoch': nan, 'best_val_macro_f1': 0.3044, 'accuracy': 0.3204, 'macro_f1': 0.3044, 'weighted_f1': 0.3035, 'uar': 0.3157}
TEST: {'best_epoch': nan, 'best_val_macro_f1': 0.3044, 'accuracy': 0.273, 'macro_f1': 0.1974, 'weighted_f1': 0.206, 'uar': 0.2584}

Training | test=emodb | model=Handcrafted SVM-RBF | seed=123
VAL : {'best_epoch': nan, 'best_val_macro_f1': 0.3044, 'accuracy': 0.3204, 'macro_f1': 0.3044, 'weighted_f1': 0.3035, 'uar': 0.3157}
TEST: {'best_epoch': nan, 'best_val_macro_f1': 0.3044, 'accuracy': 0.273, 'macro_f1': 0.1974, 'weighted_f1': 0.206, 'uar': 0.2584}

Training | test=emodb | model=Handcrafted SVM-RBF | seed=2024
VAL : {'best_epoch': nan, 'best_val_macro_f1': 0.3044, 'accuracy': 0.3204, 'macro_f1': 0.3044, 'weighted_f1': 0.3035, 'uar': 0.3157}
TEST: {'best_epoch': nan, 'best_val_macro_f1': 0.3044, 'accu

,model,test_dataset,seed,split,best_epoch,best_val_macro_f1,accuracy,macro_f1,weighted_f1,uar
0,Handcrafted SVM-RBF,emodb,42,val,NaN,0.304393,0.320442,0.304393,0.303519,0.315718
1,Handcrafted SVM-RBF,emodb,42,test,NaN,0.304393,0.272981,0.197357,0.205987,0.258379
2,Handcrafted SVM-RBF,emodb,123,val,NaN,0.304393,0.320442,0.304393,0.303519,0.315718
3,Handcrafted SVM-RBF,emodb,123,test,NaN,0.304393,0.272981,0.197357,0.205987,0.258379
4,Handcrafted SVM-RBF,emodb,2024,val,NaN,0.304393,0.320442,0.304393,0.303519,0.315718
5,Handcrafted SVM-RBF,emodb,2024,test,NaN,0.304393,0.272981,0.197357,0.205987,0.258379
6,Handcrafted SVM-RBF,ravdess,42,val,NaN,0.334797,0.315175,0.334797,0.322215,0.321264
7,Handcrafted SVM-RBF,ravdess,42,test,NaN,0.334797,0.221591,0.141564,0.154434,0.203125
8,Handcrafted SVM-RBF,ravdess,123,val,NaN,0.334797,0.315175,0.334797,0.322215,0.321264
9,Handcrafted SVM-RBF,ravdess,123,test,NaN,0.334797,0.221591,0.141564,0.154434,0.203125


Saved: /content/drive/MyDrive/New Jurnal Cross/results_lodo_utterance_fusion_plus_base/lodo_svm_results.csv


## 11. Combine all utterance-level LODO results

In [13]:
results_all = pd.concat([mlp_results, svm_results], ignore_index=True)
results_all.to_csv(OUT_DIR / "lodo_all_seed_results_with_handcrafted.csv", index=False)

# Final all-model filename used by downstream plots/tables
results_all.to_csv(OUT_DIR / "lodo_all_seed_results_final.csv", index=False)

display(results_all)
print("Saved:", OUT_DIR / "lodo_all_seed_results_with_handcrafted.csv")
print("Saved:", OUT_DIR / "lodo_all_seed_results_final.csv")

,model,test_dataset,seed,split,best_epoch,best_val_macro_f1,accuracy,macro_f1,weighted_f1,uar
0,emotion2vec MLP,emodb,42,val,19.0,0.780625,0.781768,0.780625,0.782904,0.786041
1,emotion2vec MLP,emodb,42,test,19.0,0.780625,0.835655,0.831243,0.831998,0.833800
2,emotion2vec MLP,emodb,123,val,6.0,0.786820,0.787293,0.786820,0.786547,0.789345
3,emotion2vec MLP,emodb,123,test,6.0,0.786820,0.845404,0.841127,0.842525,0.843851
4,emotion2vec MLP,emodb,2024,val,2.0,0.780464,0.781768,0.780464,0.783300,0.785326
...,...,...,...,...,...,...,...,...,...,...
67,Handcrafted SVM-RBF,resd,42,test,NaN,0.453857,0.234558,0.225884,0.223193,0.245994
68,Handcrafted SVM-RBF,resd,123,val,NaN,0.453857,0.461538,0.453857,0.445465,0.477807
69,Handcrafted SVM-RBF,resd,123,test,NaN,0.453857,0.234558,0.225884,0.223193,0.245994
70,Handcrafted SVM-RBF,resd,2024,val,NaN,0.453857,0.461538,0.453857,0.445465,0.477807


Saved: /content/drive/MyDrive/New Jurnal Cross/results_lodo_utterance_fusion_plus_base/lodo_all_seed_results_with_handcrafted.csv
Saved: /content/drive/MyDrive/New Jurnal Cross/results_lodo_utterance_fusion_plus_base/lodo_all_seed_results_final.csv


## 12. Summary tables for manuscript and supplementary materials

In [14]:
metrics = ["accuracy", "macro_f1", "weighted_f1", "uar"]

MODELS_ALL = [
    "Handcrafted SVM-RBF",
    "Handcrafted MLP",
    "emotion2vec MLP",
    "Concat Fusion MLP",
]

summary_rows = []

for model_name in MODELS_ALL:
    for test_dataset in DATASETS:
        for split in ["val", "test"]:
            sub = results_all[
                (results_all["model"] == model_name) &
                (results_all["test_dataset"] == test_dataset) &
                (results_all["split"] == split)
            ]

            row = {
                "model": model_name,
                "test_dataset": test_dataset,
                "split": split,
                "n_seeds": len(sub),
                "best_epoch_mean": sub["best_epoch"].mean(),
                "best_epoch_std": sub["best_epoch"].std(ddof=1),
                "best_val_macro_f1_mean": sub["best_val_macro_f1"].mean(),
                "best_val_macro_f1_std": sub["best_val_macro_f1"].std(ddof=1),
            }

            for metric in metrics:
                row[f"{metric}_mean"] = sub[metric].mean()
                row[f"{metric}_std"] = sub[metric].std(ddof=1)

            summary_rows.append(row)

summary = pd.DataFrame(summary_rows)
summary.to_csv(OUT_DIR / "lodo_summary_mean_std.csv", index=False)

display(summary)
print("Saved:", OUT_DIR / "lodo_summary_mean_std.csv")

,model,test_dataset,split,n_seeds,best_epoch_mean,best_epoch_std,best_val_macro_f1_mean,best_val_macro_f1_std,accuracy_mean,accuracy_std,macro_f1_mean,macro_f1_std,weighted_f1_mean,weighted_f1_std,uar_mean,uar_std
0,Handcrafted SVM-RBF,emodb,val,3,NaN,NaN,0.304393,0.000000,0.320442,0.000000e+00,0.304393,0.000000,0.303519,0.000000,0.315718,0.000000
1,Handcrafted SVM-RBF,emodb,test,3,NaN,NaN,0.304393,0.000000,0.272981,0.000000e+00,0.197357,0.000000,0.205987,0.000000,0.258379,0.000000
2,Handcrafted SVM-RBF,ravdess,val,3,NaN,NaN,0.334797,0.000000,0.315175,0.000000e+00,0.334797,0.000000,0.322215,0.000000,0.321264,0.000000
3,Handcrafted SVM-RBF,ravdess,test,3,NaN,NaN,0.334797,0.000000,0.221591,0.000000e+00,0.141564,0.000000,0.154434,0.000000,0.203125,0.000000
4,Handcrafted SVM-RBF,resd,val,3,NaN,NaN,0.453857,0.000000,0.461538,6.798700e-17,0.453857,0.000000,0.445465,0.000000,0.477807,0.000000
5,Handcrafted SVM-RBF,resd,test,3,NaN,NaN,0.453857,0.000000,0.234558,3.399350e-17,0.225884,0.000000,0.223193,0.000000,0.245994,0.000000
6,Handcrafted MLP,emodb,val,3,4.000000,2.645751,0.332095,0.002220,0.338858,4.219683e-03,0.332095,0.002220,0.330533,0.001642,0.341102,0.006309
7,Handcrafted MLP,emodb,test,3,4.000000,2.645751,0.332095,0.002220,0.285051,8.509890e-03,0.197985,0.014324,0.203113,0.015878,0.270249,0.011317
8,Handcrafted MLP,ravdess,val,3,6.000000,1.000000,0.332948,0.006091,0.334630,6.739497e-03,0.332948,0.006091,0.327586,0.003542,0.335698,0.007881
9,Handcrafted MLP,ravdess,test,3,6.000000,1.000000,0.332948,0.006091,0.259470,2.553308e-02,0.162702,0.031090,0.177494,0.033917,0.237847,0.023405


Saved: /content/drive/MyDrive/New Jurnal Cross/results_lodo_utterance_fusion_plus_base/lodo_summary_mean_std.csv


In [15]:
paper_rows = []

for model_name in MODELS_ALL:
    for test_dataset in DATASETS:
        sub = summary[
            (summary["model"] == model_name) &
            (summary["test_dataset"] == test_dataset) &
            (summary["split"] == "test")
        ].iloc[0]

        paper_rows.append({
            "Model": model_name,
            "Held-out Test Dataset": test_dataset.upper(),
            "Accuracy": mean_std_str(sub["accuracy_mean"], sub["accuracy_std"]),
            "UAR": mean_std_str(sub["uar_mean"], sub["uar_std"]),
            "Macro-F1": mean_std_str(sub["macro_f1_mean"], sub["macro_f1_std"]),
            "Weighted-F1": mean_std_str(sub["weighted_f1_mean"], sub["weighted_f1_std"]),
        })

paper_table = pd.DataFrame(paper_rows)
paper_table.to_csv(OUT_DIR / "lodo_paper_table_test.csv", index=False)

display(paper_table)
print("Saved:", OUT_DIR / "lodo_paper_table_test.csv")

,Model,Held-out Test Dataset,Accuracy,UAR,Macro-F1,Weighted-F1
0,Handcrafted SVM-RBF,EMODB,27.30 ± 0.00,25.84 ± 0.00,19.74 ± 0.00,20.60 ± 0.00
1,Handcrafted SVM-RBF,RAVDESS,22.16 ± 0.00,20.31 ± 0.00,14.16 ± 0.00,15.44 ± 0.00
2,Handcrafted SVM-RBF,RESD,23.46 ± 0.00,24.60 ± 0.00,22.59 ± 0.00,22.32 ± 0.00
3,Handcrafted MLP,EMODB,28.51 ± 0.85,27.02 ± 1.13,19.80 ± 1.43,20.31 ± 1.59
4,Handcrafted MLP,RAVDESS,25.95 ± 2.55,23.78 ± 2.34,16.27 ± 3.11,17.75 ± 3.39
5,Handcrafted MLP,RESD,23.76 ± 0.05,24.61 ± 0.34,22.77 ± 0.71,22.75 ± 0.84
6,emotion2vec MLP,EMODB,83.98 ± 0.50,83.81 ± 0.52,83.52 ± 0.52,83.63 ± 0.55
7,emotion2vec MLP,RAVDESS,94.73 ± 0.39,94.99 ± 0.28,94.52 ± 0.42,94.76 ± 0.38
8,emotion2vec MLP,RESD,59.38 ± 0.27,58.43 ± 0.41,59.31 ± 0.34,59.63 ± 0.26
9,Concat Fusion MLP,EMODB,83.19 ± 0.98,83.07 ± 1.07,82.60 ± 1.19,82.72 ± 1.12


Saved: /content/drive/MyDrive/New Jurnal Cross/results_lodo_utterance_fusion_plus_base/lodo_paper_table_test.csv


In [16]:
avg_rows = []

test_summary = summary[summary["split"] == "test"].copy()

for model_name in MODELS_ALL:
    sub = test_summary[test_summary["model"] == model_name]

    row = {
        "model": model_name,
        "n_lodo_folds": len(sub),
    }

    for metric in metrics:
        row[f"{metric}_mean_across_folds"] = sub[f"{metric}_mean"].mean()
        row[f"{metric}_std_across_folds"] = sub[f"{metric}_mean"].std(ddof=1)

    avg_rows.append(row)

avg_lodo = pd.DataFrame(avg_rows)
avg_lodo.to_csv(OUT_DIR / "lodo_average_across_folds.csv", index=False)

display(avg_lodo)
print("Saved:", OUT_DIR / "lodo_average_across_folds.csv")

,model,n_lodo_folds,accuracy_mean_across_folds,accuracy_std_across_folds,macro_f1_mean_across_folds,macro_f1_std_across_folds,weighted_f1_mean_across_folds,weighted_f1_std_across_folds,uar_mean_across_folds,uar_std_across_folds
0,Handcrafted SVM-RBF,3,0.243043,0.026725,0.188269,0.042888,0.194538,0.035781,0.235833,0.028995
1,Handcrafted MLP,3,0.260713,0.023741,0.196119,0.032523,0.202714,0.025024,0.251387,0.016844
2,emotion2vec MLP,3,0.793629,0.181232,0.791176,0.180106,0.793408,0.179515,0.790764,0.187360
3,Concat Fusion MLP,3,0.780598,0.181169,0.776295,0.178735,0.780052,0.178651,0.775415,0.179972


Saved: /content/drive/MyDrive/New Jurnal Cross/results_lodo_utterance_fusion_plus_base/lodo_average_across_folds.csv


## 13. Per-class metrics

In [17]:
per_class_rows = []

for model_name in MODELS_ALL:
    safe_model_name = model_name.lower().replace(" ", "_").replace("-", "_")

    for test_dataset in DATASETS:
        for seed in SEEDS:
            report_path = (
                OUT_DIR
                / safe_model_name
                / f"test_{test_dataset}"
                / f"seed_{seed}"
                / "test_classification_report.csv"
            )

            if not report_path.exists():
                raise FileNotFoundError(f"Missing report: {report_path}")

            report = pd.read_csv(report_path, index_col=0)

            for label in LABELS:
                per_class_rows.append({
                    "model": model_name,
                    "heldout_dataset": test_dataset,
                    "seed": seed,
                    "class": label,
                    "precision": report.loc[label, "precision"],
                    "recall": report.loc[label, "recall"],
                    "f1": report.loc[label, "f1-score"],
                    "support": report.loc[label, "support"],
                })

per_class_df = pd.DataFrame(per_class_rows)
per_class_df.to_csv(OUT_DIR / "lodo_per_class_all_seeds.csv", index=False)

per_class_summary = (
    per_class_df
    .groupby(["model", "heldout_dataset", "class"])
    .agg(
        precision_mean=("precision", "mean"),
        precision_std=("precision", "std"),
        recall_mean=("recall", "mean"),
        recall_std=("recall", "std"),
        f1_mean=("f1", "mean"),
        f1_std=("f1", "std"),
        support_mean=("support", "mean"),
    )
    .reset_index()
)

per_class_summary.to_csv(
    OUT_DIR / "lodo_per_class_summary_mean_std.csv",
    index=False,
)

display(per_class_summary)
print("Saved:", OUT_DIR / "lodo_per_class_all_seeds.csv")
print("Saved:", OUT_DIR / "lodo_per_class_summary_mean_std.csv")

,model,heldout_dataset,class,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,support_mean
0,Concat Fusion MLP,emodb,angry,0.793832,0.002933,0.980952,0.010911,0.877513,0.005406,140.0
1,Concat Fusion MLP,emodb,disgust,0.808504,0.004319,0.823899,0.044583,0.815799,0.024282,106.0
2,Concat Fusion MLP,emodb,fear,0.934639,0.032896,0.596206,0.046230,0.726541,0.025405,123.0
3,Concat Fusion MLP,emodb,happy,0.857145,0.040089,0.765537,0.032084,0.807747,0.008745,118.0
4,Concat Fusion MLP,emodb,neutral,0.765396,0.042899,0.955975,0.014411,0.849656,0.026325,106.0
...,...,...,...,...,...,...,...,...,...,...
67,emotion2vec MLP,resd,disgust,0.629310,0.022715,0.600000,0.023562,0.613794,0.007666,185.0
68,emotion2vec MLP,resd,fear,0.422252,0.004325,0.847534,0.013453,0.563625,0.002681,223.0
69,emotion2vec MLP,resd,happy,0.677202,0.027686,0.600917,0.004587,0.636593,0.012478,218.0
70,emotion2vec MLP,resd,neutral,0.640750,0.034431,0.577661,0.013176,0.607001,0.010049,191.0


Saved: /content/drive/MyDrive/New Jurnal Cross/results_lodo_utterance_fusion_plus_base/lodo_per_class_all_seeds.csv
Saved: /content/drive/MyDrive/New Jurnal Cross/results_lodo_utterance_fusion_plus_base/lodo_per_class_summary_mean_std.csv


In [18]:
paper_class_rows = []

for model_name in MODELS_ALL:
    for test_dataset in DATASETS:
        sub = per_class_summary[
            (per_class_summary["model"] == model_name) &
            (per_class_summary["heldout_dataset"] == test_dataset)
        ]

        row = {
            "Model": model_name,
            "Held-out Dataset": test_dataset.upper(),
        }

        for label in LABELS:
            label_row = sub[sub["class"] == label].iloc[0]
            row[label] = mean_std_str(
                label_row["f1_mean"],
                label_row["f1_std"],
            )

        paper_class_rows.append(row)

paper_per_class_f1 = pd.DataFrame(paper_class_rows)
paper_per_class_f1.to_csv(
    OUT_DIR / "lodo_paper_table_per_class_f1.csv",
    index=False,
)

display(paper_per_class_f1)
print("Saved:", OUT_DIR / "lodo_paper_table_per_class_f1.csv")

,Model,Held-out Dataset,angry,disgust,fear,happy,neutral,sad
0,Handcrafted SVM-RBF,EMODB,56.34 ± 0.00,27.96 ± 0.00,8.42 ± 0.00,9.95 ± 0.00,12.59 ± 0.00,3.15 ± 0.00
1,Handcrafted SVM-RBF,RAVDESS,34.91 ± 0.00,21.76 ± 0.00,25.23 ± 0.00,1.00 ± 0.00,0.00 ± 0.00,2.04 ± 0.00
2,Handcrafted SVM-RBF,RESD,30.94 ± 0.00,31.64 ± 0.00,19.20 ± 0.00,10.75 ± 0.00,17.86 ± 0.00,25.14 ± 0.00
3,Handcrafted MLP,EMODB,51.14 ± 3.08,30.74 ± 3.17,5.97 ± 5.46,5.52 ± 2.85,20.94 ± 2.32,4.49 ± 5.03
4,Handcrafted MLP,RAVDESS,38.17 ± 1.18,31.66 ± 8.71,18.31 ± 3.33,0.00 ± 0.00,0.00 ± 0.00,9.48 ± 12.91
5,Handcrafted MLP,RESD,37.98 ± 3.03,26.46 ± 2.16,22.53 ± 4.13,9.35 ± 3.40,15.80 ± 5.26,24.49 ± 2.38
6,emotion2vec MLP,EMODB,88.01 ± 1.56,80.69 ± 0.75,75.52 ± 0.61,81.96 ± 1.25,87.17 ± 1.26,87.76 ± 0.14
7,emotion2vec MLP,RAVDESS,97.35 ± 0.15,97.53 ± 0.41,93.68 ± 0.91,95.58 ± 0.14,91.87 ± 1.21,91.10 ± 1.07
8,emotion2vec MLP,RESD,63.38 ± 1.86,61.38 ± 0.77,56.36 ± 0.27,63.66 ± 1.25,60.70 ± 1.00,50.41 ± 2.16
9,Concat Fusion MLP,EMODB,87.75 ± 0.54,81.58 ± 2.43,72.65 ± 2.54,80.77 ± 0.87,84.97 ± 2.63,87.90 ± 0.63


Saved: /content/drive/MyDrive/New Jurnal Cross/results_lodo_utterance_fusion_plus_base/lodo_paper_table_per_class_f1.csv


## 14. Final output checklist

In [19]:
expected_outputs = [
    "training_environment_manifest.json",
    "input_feature_file_manifest_sha256.csv",
    "emotion2vec_extraction_environment_manifest.json",
    "lodo_split_manifest.csv",
    "lodo_split_summary.csv",
    "lodo_leakage_prevention_checklist.csv",
    "lodo_partition_overlap_check.csv",
    "lodo_mlp_all_seed_results.csv",
    "lodo_svm_results.csv",
    "lodo_all_seed_results_with_handcrafted.csv",
    "lodo_all_seed_results_final.csv",
    "lodo_summary_mean_std.csv",
    "lodo_paper_table_test.csv",
    "lodo_average_across_folds.csv",
    "lodo_per_class_all_seeds.csv",
    "lodo_per_class_summary_mean_std.csv",
    "lodo_paper_table_per_class_f1.csv",
]

for name in expected_outputs:
    path = OUT_DIR / name
    print(f"{name}: {'OK' if path.exists() else 'MISSING'}")

print("\nOutput directory:", OUT_DIR)

training_environment_manifest.json: OK
input_feature_file_manifest_sha256.csv: OK
emotion2vec_extraction_environment_manifest.json: OK
lodo_split_manifest.csv: OK
lodo_split_summary.csv: OK
lodo_leakage_prevention_checklist.csv: OK
lodo_partition_overlap_check.csv: OK
lodo_mlp_all_seed_results.csv: OK
lodo_svm_results.csv: OK
lodo_all_seed_results_with_handcrafted.csv: OK
lodo_all_seed_results_final.csv: OK
lodo_summary_mean_std.csv: OK
lodo_paper_table_test.csv: OK
lodo_average_across_folds.csv: OK
lodo_per_class_all_seeds.csv: OK
lodo_per_class_summary_mean_std.csv: OK
lodo_paper_table_per_class_f1.csv: OK

Output directory: /content/drive/MyDrive/New Jurnal Cross/results_lodo_utterance_fusion_plus_base
